# 03 · Task 1: the loan-amount model (step 9)

**Owner:** Alice. **Decision record:** [ADR 0018](../docs/adr/0018-loan-amount-model.md). **Plain write-up:** [learnings/09](../learnings/09_regression.md). **Shared code:** `src/features.py` (clues, tests) and `src/amounts.py` (the final model and the loan limit, used by the scorer).

**The question.** For each applicant, how much money is a sensible loan? NkwaPay needs this to set a **limit** (the most it will lend) and to plan its **reserves** (how much cash to keep ready).

**What we predict.** `Total_Amount`, the amount sent to the borrower, from the same 13 clues the default model uses (`FEATURES`, [ADR 0012](../docs/adr/0012-final-feature-set.md)). The amount itself is never a clue.

**Team rules for this notebook**
1. **Explain everything as plainly as possible.** Every step gets a markdown cell: what, why, and what the result means.
2. **Test every reasonable alternative**, side by side in a table, and pick the winner with numbers.
3. **It must run top to bottom** from a fresh clone with `data/` in place (about 10 minutes).

**Privacy:** every output is a count, a summary or a score. No borrower's row is printed. The residual plot shows density (how many loans fall in each small hexagon), not individual loans.

In [1]:
from pathlib import Path
import sys, time, warnings
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from lightgbm import LGBMRegressor
from src.features import build, FEATURES, preprocessor, splits
from src.amounts import usd_rate, AMOUNT_CLUES

TABLES, FIGURES = ROOT / "learnings" / "tables", ROOT / "learnings" / "figures"
LOCAL = ROOT / "data" / "experiments" / "step9"; LOCAL.mkdir(parents=True, exist_ok=True)
tr, te = build(str(ROOT / "data" / "zindi"))
amt = tr["Total_Amount"].to_numpy(float)
ylog = np.log(amt)
print(len(tr), "labelled Kenyan loans;", len(te), "Zindi test loans")

68464 labelled Kenyan loans; 18594 Zindi test loans


## 1. The target: why we predict the *log* of the amount

Loan amounts are very uneven. Most loans are a few thousand shillings, but a handful are millions. If a model learns the raw amount, those few giant loans dominate everything it learns.

The **log** turns "times" into "plus": a loan 10× bigger is just +2.3 on the log scale, whatever its size. So 1,000 → 10,000 and 100,000 → 1,000,000 count as the same step. **Skewness** measures lopsidedness (0 = balanced).

In [2]:
a = pd.Series(amt)
print(f"smallest {a.min():,.0f} | 25% {a.quantile(.25):,.0f} | middle {a.median():,.0f} | 75% {a.quantile(.75):,.0f} | "
      f"average {a.mean():,.0f} | largest {a.max():,.0f} (KES)")
print(f"loans over 1 million KES: {(a > 1e6).sum()} of {len(a)} ({(a > 1e6).mean():.2%}), but {a[a > 1e6].sum() / a.sum():.0%} of all money lent")
print(f"skewness: raw {a.skew():.1f} -> log {np.log(a).skew():.2f}")

smallest 2 | 25% 2,289 | middle 5,249 | 75% 11,420 | average 14,763 | largest 23,000,000 (KES)
loans over 1 million KES: 46 of 68464 (0.07%), but 15% of all money lent
skewness: raw 104.3 -> log 0.13


## 2. How we score a model

Every model is learned on one part of the data and scored on loans it has never seen (the **tests** from `splits()`, [ADR 0010](../docs/adr/0010-validation-tests.md)):
- **future loans** (`time_block`): learn from older loans, predict newer ones. **This is the main test** because NkwaPay will always predict future loans;
- **future loans, new customers** (`time_new_customers`): the same, but only for customers never seen while learning. This is the closest Kenyan stand-in for Ghana;
- **new customers** (`fold_customer`): a customer's loans stay on one side;
- **shuffled** (`fold_shuffled`): the naive way, reported only to show how much it flatters.

The scores (lower is better, except "within 2×"):
- **MAE (log):** the average size of the miss on the log scale. 0.58 means a typical guess is off by a factor of e^0.58 ≈ 1.8 (too high or too low).
- **RMSE (log):** like MAE, but big misses count more (it squares them before averaging).
- **MAE / RMSE (KES):** the same on the original shilling scale, after turning the log prediction back into shillings.
- **median miss (KES):** half of the guesses are closer than this. Unlike RMSE it isn't driven by a few giant loans.
- **within 2×:** the share of loans where the guess is between half and double the real amount.
- **total ratio:** all predicted money ÷ all real money in the test part. 1.00 means the reserves estimate is right overall.

Each score is worked out per test part and then averaged.

In [3]:
def oof(make, scheme, target="log", X=None, y=None):
    '''Predictions on the log scale for every test loan of every fold (the model never saw them while learning).
    X, y: other clue values / log target (used for the dollar version); default the shilling ones.'''
    X = tr if X is None else X
    y = (ylog if target == "log" else amt) if y is None else y
    parts = []
    for f, (a, b) in enumerate(splits(tr, scheme)):
        m = make().fit(X.loc[a, FEATURES], y[a])
        p = m.predict(X.loc[b, FEATURES])
        parts.append(pd.DataFrame({"fold": f, "pred": np.log(np.clip(p, 1, None)) if target == "raw" else p}, index=b))
    return pd.concat(parts)

def scores(o, kes=None):
    '''Average of each score over the test parts. kes: predicted shillings (default exp of the log prediction).'''
    o = o.assign(kes=np.exp(o.pred) if kes is None else kes, y=ylog[o.index], a=amt[o.index])
    def one(g):
        e, d = g.y - g.pred, g.a - g.kes
        return pd.Series({"MAE log": np.abs(e).mean(), "RMSE log": np.sqrt((e ** 2).mean()),
                          "MAE KES": np.abs(d).mean(), "RMSE KES": np.sqrt((d ** 2).mean()),
                          "median miss KES": np.abs(d).median(), "within 2x": (np.abs(e) < np.log(2)).mean(),
                          "total ratio": g.kes.sum() / g.a.sum()})
    return o.groupby("fold").apply(one).mean()

class Sampled:
    '''Learn on a random sample of the learning part (SVR gets slow above ~10,000 rows).'''
    def __init__(self, model, n): self.model, self.n = model, n
    def fit(self, X, y):
        i = np.random.default_rng(0).choice(len(X), min(self.n, len(X)), replace=False)
        self.model.fit(X.iloc[i], np.asarray(y)[i]); return self
    def predict(self, X): return self.model.predict(X)

P = lambda m: make_pipeline(preprocessor(FEATURES), m)   # the shared preparation always comes first

## 3. Settings for each model

Most models have a knob. We try a few values of each and keep the best, judged on the **new customers** test. We pick settings there, not on future loans, so the main test stays untouched for the final comparison.

- **Ridge / Lasso:** how hard the weights are pulled towards zero (`alpha`). Lasso can push weights to exactly zero.
- **k-NN:** how many similar past loans to average (`k`).
- **Decision tree:** how many questions deep it may go.
- **Random forest:** the smallest group of loans a leaf may hold (bigger = smoother).
- **LightGBM:** how many trees, and how many leaves each.
- **SVR:** how much it is allowed to bend to fit (`C`), learned on a 5,000-loan sample (it is too slow on all 55,000).

In [4]:
GRID = {
    "Ridge": {f"alpha={v}": (lambda v=v: P(Ridge(v))) for v in [0.1, 1, 10, 100]},
    "Lasso": {f"alpha={v}": (lambda v=v: P(Lasso(v))) for v in [0.0001, 0.001, 0.01]},
    "k-NN": {f"k={v}": (lambda v=v: P(KNeighborsRegressor(v))) for v in [5, 25, 100]},
    "Decision tree": {f"depth={v}": (lambda v=v: P(DecisionTreeRegressor(max_depth=v, min_samples_leaf=50, random_state=0))) for v in [4, 8, 12]},
    "Random forest": {f"leaf={v}": (lambda v=v: P(RandomForestRegressor(200, min_samples_leaf=v, max_features=0.5, n_jobs=-1, random_state=0))) for v in [5, 20, 50]},
    "LightGBM": {f"{n} trees, {l} leaves": (lambda n=n, l=l: P(LGBMRegressor(n_estimators=n, learning_rate=0.05, num_leaves=l, verbose=-1, random_state=0)))
                 for n, l in [(300, 15), (300, 31), (600, 63)]},
    "SVR": {f"C={v}": (lambda v=v: Sampled(P(SVR(C=v, epsilon=0.1)), 5000)) for v in [0.3, 1, 3]},
}
rows = []
for model, opts in GRID.items():
    for setting, make in opts.items():
        rows.append({"model": model, "setting": setting, "MAE log (new customers)": scores(oof(make, "fold_customer"))["MAE log"]})
tune = pd.DataFrame(rows)
best = tune.loc[tune.groupby("model")["MAE log (new customers)"].idxmin()].set_index("model")["setting"]
tune["chosen"] = tune.apply(lambda r: "yes" if best[r.model] == r.setting else "", axis=1)
tune.to_csv(TABLES / "09_settings.csv", index=False)
tune.round(4)

,model,setting,MAE log (new customers),chosen
0,Ridge,alpha=0.1,0.5966,yes
1,Ridge,alpha=1,0.5966,
2,Ridge,alpha=10,0.5970,
3,Ridge,alpha=100,0.5977,
4,Lasso,alpha=0.0001,0.5967,yes
5,Lasso,alpha=0.001,0.5978,
6,Lasso,alpha=0.01,0.6026,
7,k-NN,k=5,0.6458,
8,k-NN,k=25,0.5983,
9,k-NN,k=100,0.5953,yes


## 4. Every model on every test

Now the chosen setting of each model, plus:
- **guess the average** (the mean baseline: always predict the average log amount);
- **OLS**, plain linear regression with no knob. Its weights are the "normal equations" solution, see `knowledge/derivations.md`.

In [5]:
MODELS = {"Guess the average": lambda: DummyRegressor(), "OLS (linear regression)": lambda: P(LinearRegression())}
MODELS.update({m: GRID[m][best[m]] for m in GRID})
SCHEMES = {"future loans": "time_block", "future loans, new customers": "time_new_customers",
           "new customers": "fold_customer", "shuffled": "fold_shuffled"}
rows, OOF = [], {}
for name, make in MODELS.items():
    t = time.time()
    for test, scheme in SCHEMES.items():
        o = oof(make, scheme); OOF[name, scheme] = o
        rows.append({"model": name, "test": test, **scores(o)})
    print(f"{name}: {time.time() - t:.0f}s")
main = pd.DataFrame(rows)
main.to_csv(TABLES / "09_models.csv", index=False)
main[main.test == "future loans"].drop(columns="test").set_index("model").round(3)

Guess the average: 0s


OLS (linear regression): 2s


Ridge: 2s


Lasso: 2s


k-NN: 8s


Decision tree: 4s


Random forest: 37s


LightGBM: 41s


SVR: 46s


,MAE log,RMSE log,MAE KES,RMSE KES,median miss KES,within 2x,total ratio
model,,,,,,,
Guess the average,0.972,1.262,12148.578,110139.168,3641.785,0.441,0.436
OLS (linear regression),0.630,0.822,13834.081,167857.516,2269.075,0.644,0.914
Ridge,0.630,0.823,13802.324,167314.019,2269.491,0.644,0.913
Lasso,0.628,0.820,12860.338,152289.955,2261.920,0.646,0.873
k-NN,0.603,0.818,9189.429,106751.624,2116.834,0.682,0.558
Decision tree,0.595,0.810,8746.581,105522.469,2145.777,0.689,0.639
Random forest,0.583,0.795,8393.292,103648.020,2110.288,0.697,0.657
LightGBM,0.581,0.796,8462.613,103473.781,2086.969,0.702,0.688
SVR,0.592,0.860,9111.521,106291.094,2130.073,0.708,0.718


**The same models on the other tests** (MAE log only). The "inflation" column shows how much better the naive shuffled test makes each model look than future loans.

In [6]:
wide = main.pivot(index="model", columns="test", values="MAE log")[list(SCHEMES)]
wide["shuffled flatters by"] = wide["future loans"] - wide["shuffled"]
wide.loc[list(MODELS)].round(3)

test,future loans,"future loans, new customers",new customers,shuffled,shuffled flatters by
model,,,,,
Guess the average,0.972,1.045,0.978,0.977,-0.005
OLS (linear regression),0.630,0.694,0.597,0.596,0.034
Ridge,0.630,0.694,0.597,0.596,0.034
Lasso,0.628,0.687,0.597,0.596,0.031
k-NN,0.603,0.647,0.595,0.593,0.010
Decision tree,0.595,0.634,0.588,0.585,0.009
Random forest,0.583,0.596,0.580,0.577,0.005
LightGBM,0.581,0.593,0.579,0.576,0.005
SVR,0.592,0.609,0.570,0.568,0.024


## 5. Raw amount vs log amount as the target

The other way would be to learn the raw shilling amount directly. We try it with three models and score both versions on both scales.

In [7]:
rows = []
for name in ["Ridge", "Random forest", "LightGBM"]:
    for target in ["log", "raw"]:
        o = OOF[name, "time_block"] if target == "log" else oof(MODELS[name], "time_block", target="raw")
        rows.append({"model": name, "target learned": target, **scores(o)})
raw_vs_log = pd.DataFrame(rows); raw_vs_log.to_csv(TABLES / "09_raw_vs_log.csv", index=False)
raw_vs_log.round(3)

,model,target learned,MAE log,RMSE log,MAE KES,RMSE KES,median miss KES,within 2x,total ratio
0,Ridge,log,0.630,0.823,13802.324,167314.019,2269.491,0.644,0.913
1,Ridge,raw,2.194,3.510,11634.486,103638.397,4702.401,0.443,1.118
2,Random forest,log,0.583,0.795,8393.292,103648.020,2110.288,0.697,0.657
3,Random forest,raw,0.593,0.874,8610.833,102319.311,2195.591,0.716,0.955
4,LightGBM,log,0.581,0.796,8462.613,103473.781,2086.969,0.702,0.688
5,LightGBM,raw,0.639,1.000,9085.210,103432.497,2268.737,0.698,0.946


## 6. Turning a log prediction back into shillings

The exponent of a log prediction, e^pred, gives the **typical** (median) amount. But the **average** amount is higher, because amounts are lopsided: a few big loans pull the average up. The **smearing** fix (Duan, 1983) multiplies by the average of e^(learning errors). That gives an estimate of the average.

Which to use depends on the job:
- **typical** is right for a single borrower's limit;
- **average** is right for adding up reserves.

In [8]:
rows = []
for name in ["Ridge", "LightGBM", "Random forest"]:
    o = OOF[name, "time_block"]; smear = pd.Series(index=o.index, dtype=float)
    for f, (a, b) in enumerate(splits(tr, "time_block")):
        m = MODELS[name]().fit(tr.loc[a, FEATURES], ylog[a])
        smear[b] = np.mean(np.exp(ylog[a] - m.predict(tr.loc[a, FEATURES])))
    for how, kes in [("typical: e^pred", np.exp(o.pred)), ("average: e^pred x smearing", np.exp(o.pred) * smear)]:
        rows.append({"model": name, "back to KES": how, "smearing factor": 1.0 if how.startswith("typ") else smear.mean(), **scores(o, kes)})
back = pd.DataFrame(rows); back.to_csv(TABLES / "09_back_transform.csv", index=False)
back.round(3)

,model,back to KES,smearing factor,MAE log,RMSE log,MAE KES,RMSE KES,median miss KES,within 2x,total ratio
0,Ridge,typical: e^pred,1.000,0.630,0.823,13802.324,167314.019,2269.491,0.644,0.913
1,Ridge,average: e^pred x smearing,1.465,0.630,0.823,17104.892,216383.675,2286.743,0.644,1.311
2,LightGBM,typical: e^pred,1.000,0.581,0.796,8462.613,103473.781,2086.969,0.702,0.688
3,LightGBM,average: e^pred x smearing,1.424,0.581,0.796,8895.570,102702.669,2350.892,0.702,0.982
4,Random forest,typical: e^pred,1.000,0.583,0.795,8393.292,103648.020,2110.288,0.697,0.657
5,Random forest,average: e^pred x smearing,1.376,0.583,0.795,8444.943,102424.548,2220.435,0.697,0.906


## 7. Local currency (KES) vs US dollars

All Kenyan amounts are in shillings, and the shilling lost value over these years (110 → 140 per dollar). Converting to dollars could make loans from different years more alike. We convert the target and the two past-amount clues (`mean_prev_amount`, `last_amount`) using that year's average exchange rate. 2024 has no published rate yet, so it uses 2023's. We predict in dollars, then convert back to shillings to score.

This matters much more for Ghana (step 10), where amounts are in cedis.

In [9]:
rate = usd_rate(tr, ROOT / "data" / "zindi")          # local currency per US dollar, row by row
tr_usd = tr.copy(); tr_usd[AMOUNT_CLUES] = tr[AMOUNT_CLUES].div(rate, axis=0)
rows = []
for name in ["Ridge", "LightGBM"]:
    for cur in ["KES", "USD"]:
        for test, scheme in [("future loans", "time_block"), ("future loans, new customers", "time_new_customers")]:
            if cur == "KES":
                o = OOF[name, scheme]
            else:  # learn log(dollars), then add back log(rate) to score in shillings
                o = oof(MODELS[name], scheme, X=tr_usd, y=ylog - np.log(rate))
                o["pred"] += np.log(rate[o.index])
            rows.append({"model": name, "currency": cur, "test": test, **scores(o)})
cur = pd.DataFrame(rows); cur.to_csv(TABLES / "09_currency.csv", index=False)
cur.round(3)

,model,currency,test,MAE log,RMSE log,MAE KES,RMSE KES,median miss KES,within 2x,total ratio
0,Ridge,KES,future loans,0.630,0.823,13802.324,167314.019,2269.491,0.644,0.913
1,Ridge,KES,"future loans, new customers",0.694,0.904,34372.771,347676.856,2245.843,0.587,0.946
2,Ridge,USD,future loans,0.631,0.824,14548.132,181303.286,2273.247,0.642,0.948
3,Ridge,USD,"future loans, new customers",0.692,0.905,36878.393,374021.717,2244.185,0.587,0.999
4,LightGBM,KES,future loans,0.581,0.796,8462.613,103473.781,2086.969,0.702,0.688
5,LightGBM,KES,"future loans, new customers",0.593,0.833,17910.728,233118.703,1731.129,0.694,0.533
6,LightGBM,USD,future loans,0.586,0.799,8488.522,103245.261,2131.360,0.695,0.687
7,LightGBM,USD,"future loans, new customers",0.597,0.835,17855.871,232660.984,1750.912,0.686,0.542


## 8. Is the winner really better? (bootstrap)

The differences in section 4 are small, so we check whether they are bigger than luck. A **bootstrap** re-draws the test loans many times (here 1,000) and recomputes the difference each time. The middle 95% of those differences is the **95% interval**. If it doesn't include 0, the difference is real.

We re-draw whole **customers**, not single loans, because one customer's loans are alike and aren't independent evidence. A negative difference means the first model misses less.

In [10]:
def cluster_boot(d, groups, n=1000, seed=0):
    g = pd.DataFrame({"d": d, "c": groups}).groupby("c")["d"].agg(["sum", "size"])
    s, k, rng, out = g["sum"].to_numpy(), g["size"].to_numpy(), np.random.default_rng(seed), []
    for _ in range(n // 100):
        i = rng.integers(0, len(g), (100, len(g)))
        out.append(s[i].sum(1) / k[i].sum(1))
    return np.percentile(np.concatenate(out), [2.5, 97.5])

err = lambda name: np.abs(ylog[OOF[name, "time_block"].index] - OOF[name, "time_block"].pred)
pairs = [("Random forest", "LightGBM"), ("Random forest", "SVR"), ("LightGBM", "SVR"), ("LightGBM", "Ridge"),
         ("LightGBM", "Lasso"), ("Random forest", "Lasso"), ("Lasso", "Ridge"), ("Lasso", "OLS (linear regression)"),
         ("Decision tree", "Lasso"), ("Ridge", "Guess the average")]
cust = tr.loc[OOF["Ridge", "time_block"].index, "customer_id"].to_numpy()
rows = []
for a_, b_ in pairs:
    d = (err(a_) - err(b_)).to_numpy(); lo, hi = cluster_boot(d, cust)
    rows.append({"A": a_, "B": b_, "A minus B (MAE log)": d.mean(), "95% low": lo, "95% high": hi,
                 "verdict": "A better" if hi < 0 else ("B better" if lo > 0 else "a tie")})
boot = pd.DataFrame(rows); boot.to_csv(TABLES / "09_bootstrap.csv", index=False)
boot.round(4)

,A,B,A minus B (MAE log),95% low,95% high,verdict
0,Random forest,LightGBM,0.0020,0.0004,0.0034,B better
1,Random forest,SVR,-0.0098,-0.0153,-0.0047,A better
2,LightGBM,SVR,-0.0117,-0.0172,-0.0067,A better
3,LightGBM,Ridge,-0.0493,-0.0540,-0.0446,A better
4,LightGBM,Lasso,-0.0468,-0.0512,-0.0422,A better
5,Random forest,Lasso,-0.0449,-0.0494,-0.0402,A better
6,Lasso,Ridge,-0.0025,-0.0029,-0.0022,A better
7,Lasso,OLS (linear regression),-0.0023,-0.0026,-0.0020,A better
8,Decision tree,Lasso,-0.0330,-0.0371,-0.0287,A better
9,Ridge,Guess the average,-0.3423,-0.3607,-0.3234,A better


## 9. The choice

**We choose LightGBM, learned on the log amount in shillings, turned back with e^pred for limits and with smearing for reserves.** Why, from the tables above:

1. **It misses least on future loans.** The log error is 0.581, about 1.8× off; Ridge's is 0.630, about 1.9× off. 70% of guesses are within 2× of the real amount, against 64% for Ridge.
2. **The gain is real.** LightGBM beats Ridge by 0.049 on the log scale, and the 95% interval (−0.054 to −0.045) is far from 0. It also beats SVR and, just barely, the random forest (0.002).
3. **The log target wins.** Learning raw shillings lets a few giant loans steer the model. LightGBM on raw amounts misses more on the log scale (0.64 vs 0.58), and Ridge on raw amounts is far worse (2.19).
4. **Dollars don't help inside Kenya** (0.586 vs 0.581). We keep shillings here; step 10 tests dollars on Ghana, where it matters.
5. **Honest testing barely matters here.** Unlike the default model, the shuffled test flatters each model by only 0.005 to 0.034. The amount depends on the loan's type and the borrower's past sizes, not on recognising the customer.

**Why not the simplest model?** The committee prefers simple models, and that rule decided Task 2 (logistic regression). Here the trade is different:
- the amount model never decides *whether* someone gets a loan, and it gives no reasons to borrowers;
- its job is a sensible limit, and the tree model is clearly more accurate at it.

Ridge stays as a fallback with weights anyone can read: it is about 5% further off per guess.

In [11]:
CHOSEN = "LightGBM"

## 10. Residual diagnostics: where does the model miss?

A **residual** is real minus predicted, here on the log scale (+0.7 means the real loan was about twice the guess). In a healthy model:
- the residuals are centred on 0 at every predicted size (no curve);
- the spread doesn't change much with size.

The plot shows **density** (darker = more loans), plus the average residual in each band of predicted size. Future-loans test, chosen model.

In [12]:
o = OOF[CHOSEN, "time_block"]; fitted, resid = o.pred.to_numpy(), ylog[o.index] - o.pred.to_numpy()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
hb = ax[0].hexbin(fitted, resid, gridsize=45, bins="log", mincnt=1, cmap="Blues")
edges = np.quantile(fitted, np.linspace(0, 1, 21)); mids = (edges[1:] + edges[:-1]) / 2
band = np.clip(np.searchsorted(edges[1:-1], fitted, side="right"), 0, 19)
ax[0].plot(mids, [resid[band == i].mean() for i in range(20)], "o-", color="darkorange", label="average residual")
ax[0].axhline(0, color="k", lw=.8); ax[0].legend(loc="lower left")
ax[0].set(title=f"Residual vs fitted ({CHOSEN}, future loans)", xlabel="predicted log amount (8 ≈ 3,000 KES, 10 ≈ 22,000)", ylabel="real − predicted (log)")
plt.colorbar(hb, ax=ax[0], label="loans (log scale)")
ax[1].hist(resid, bins=80, color="steelblue"); ax[1].axvline(0, color="k", lw=.8)
ax[1].set(title="How big the misses are", xlabel="real − predicted (log): ±0.7 ≈ 2× off", ylabel="loans")
plt.tight_layout(); fig.savefig(FIGURES / "09_residuals.png", dpi=130)
print(f"average residual {resid.mean():+.3f}; spread (sd) by predicted-size third: " +
      ", ".join(f"{resid[(band >= a) & (band < b)].std():.2f}" for a, b in [(0, 7), (7, 14), (14, 20)]))

average residual -0.001; spread (sd) by predicted-size third: 0.73, 0.79, 0.88


![](../learnings/figures/09_residuals.png)

**Residuals by group.** If one group's average residual is far from 0, the model systematically under- or over-predicts its loans. That would make its limits unfair, which is an input for Liza's fairness step. Groups with at least 100 loans only.

In [13]:
d = tr.loc[o.index].assign(resid=resid)
groups = {"loan type": d.loan_type, "new vs repeat": d.New_versus_Repeat,
          "earlier loans": np.where(d.n_prev_loans == 0, "first loan", np.where(d.no_known_history == 1, "earlier, no known outcome", "earlier, known outcomes")),
          "loan length": pd.cut(d.duration, [0, 7, 30, 1000], labels=["7 days", "8-30 days", "over 30 days"]),
          "defaulted later": d.target.map({0: "repaid", 1: "defaulted"})}
rows = [{"group": g, "value": v, "loans": len(r), "average residual (log)": r.mean(), "MAE log": r.abs().mean()}
        for g, s in groups.items() for v, r in d.resid.groupby(s) if len(r) >= 100]
res_groups = pd.DataFrame(rows); res_groups.to_csv(TABLES / "09_residuals_by_group.csv", index=False)
res_groups.round(3)

,group,value,loans,average residual (log),MAE log
0,loan type,Type_1,49000,-0.007,0.594
1,loan type,Type_10,400,-0.495,0.878
2,loan type,Type_4,989,0.020,0.427
3,loan type,Type_5,800,-0.072,0.249
4,loan type,Type_6,326,0.179,0.641
5,loan type,Type_7,2654,0.106,0.421
6,loan type,Type_9,190,0.344,0.504
7,new vs repeat,New Loan,387,-0.001,0.718
8,new vs repeat,Repeat Loan,54076,-0.001,0.580
9,earlier loans,"earlier, known outcomes",50513,0.016,0.571


## 11. From a predicted amount to a loan limit

The model gives the **typical** amount similar borrowers got. To make a limit we test five rules on the future-loans test:
- **no limit:** lend whatever is asked (the baseline);
- **rule of thumb:** the borrower's last loan amount. First loans get the median first loan of the learning period;
- **model, 25% / 50% / 75%:** e^(prediction + q). Here q is chosen so that 25%, 50% or 75% of *recent* similar borrowers got at most this much. q is measured on the newest 20% of each learning period, with a model learned on the older 80%, so it is never measured on the loans it is tested on.

What we look at:
- **fits under the limit:** the share of borrowers whose real loan was within the limit;
- **money lent:** the money lent if every loan is cut to its limit, as a share of the money actually lent;
- **defaulted money:** the share of that lent money that went to loans that defaulted;
- **default rate, over vs within:** the default rate of borrowers whose real loan was above the limit vs within it. If the "over" group defaults more, the limit is catching risky over-borrowing.

In [14]:
pieces = []
for f, (a, b) in enumerate(splits(tr, "time_block")):
    A = tr.loc[a].sort_values("disbursement_date", kind="mergesort"); cut = A.disbursement_date.iloc[int(.8 * len(A))]
    old, new = A[A.disbursement_date < cut], A[A.disbursement_date >= cut]
    r = np.log(new.Total_Amount) - MODELS[CHOSEN]().fit(old[FEATURES], np.log(old.Total_Amount)).predict(new[FEATURES])
    B = tr.loc[b, ["Total_Amount", "target", "last_amount", "n_prev_loans"]].copy()
    B["pred"] = OOF[CHOSEN, "time_block"].loc[b, "pred"]
    for q in (.25, .5, .75): B[f"q{int(q * 100)}"] = np.quantile(r, q)
    B["first_median"] = A.loc[A.n_prev_loans == 0, "Total_Amount"].median()
    pieces.append(B.assign(fold=f))
L = pd.concat(pieces)
rules = {"no limit": np.full(len(L), np.inf), "rule of thumb: last loan amount": L.last_amount.fillna(L.first_median).to_numpy()}
for q in (25, 50, 75): rules[f"model, {q}%"] = np.exp(L.pred + L[f"q{q}"]).to_numpy()
rows = []
for name, lim in rules.items():
    a_, y_ = L.Total_Amount.to_numpy(), L.target.to_numpy(); lent = np.minimum(a_, lim); over = a_ > lim
    rows.append({"limit rule": name, "fits under the limit": 1 - over.mean(), "money lent (share of actual)": lent.sum() / a_.sum(),
                 "defaulted money (share of lent)": lent[y_ == 1].sum() / lent.sum(),
                 "default rate, over the limit": y_[over].mean() if over.any() else np.nan, "default rate, within": y_[~over].mean()})
limits = pd.DataFrame(rows).set_index("limit rule"); limits.to_csv(TABLES / "09_limits.csv")
limits.round(3)

,fits under the limit,money lent (share of actual),defaulted money (share of lent),"default rate, over the limit","default rate, within"
limit rule,,,,,
no limit,1.000,1.000,0.152,NaN,0.018
rule of thumb: last loan amount,0.531,0.656,0.075,0.023,0.014
"model, 25%",0.215,0.389,0.059,0.019,0.014
"model, 50%",0.484,0.578,0.058,0.019,0.017
"model, 75%",0.741,0.701,0.059,0.022,0.017


**What the limit table means.**
- **Any limit cuts the money lost to defaulters a lot.** With no limit, 15% of the money lent goes to loans that later default, even though only 1.8% of *loans* default. Defaulters tend to take bigger loans. Every limit rule brings that down to 6–8%.
- **The 75% rule beats the rule of thumb on all three measures:**
  - more borrowers fit under it (74% vs 53%);
  - more money is lent (70% vs 66%);
  - less of that money goes to defaulters (5.9% vs 7.5%).
- **The 50% rule** is a little safer (5.8%), but half of all borrowers would get less than they took. That's too tight for a product meant to widen access.
- **Borrowers who ask above the 75% limit default a bit more** (2.2% vs 1.7%). So the limit catches some risky over-borrowing.

**Decision: limit = e^(prediction + q75)**, the amount that 75% of similar recent borrowers got at most (`LIMIT_QUANTILE = 0.75` in `src/amounts.py`). The default model still decides approve or decline first (step 7). The limit only applies to approved applicants. Under Ghana's directive, the limit must also pass an affordability check, which this data can't do (we have no income data). See [ADR 0018](../docs/adr/0018-loan-amount-model.md).

## 12. The final model, saved for the scorer

`src/amounts.py` holds the final recipe, so the scorer (step 12) uses exactly this. It learns the chosen model on every labelled loan. It measures the limit offset and the smearing factor on the newest 20% of loans, with a model learned on the older 80%.

Saved **locally only** (in the ignored `data/` folder):
- the model;
- predictions for the Zindi test loans.

Kenyan test predictions are in shillings. Ghanaian ones are covered in step 10.

In [15]:
import joblib
from src.amounts import final_amount_model, LIMIT_QUANTILE
final = final_amount_model(tr)
joblib.dump(final, LOCAL / "amount_model.joblib")
ke = te[te.country_id == "Kenya"]
out = pd.DataFrame({"ID": ke.ID, "typical_amount": final.typical(ke), "limit": final.limit(ke)})
out.to_csv(LOCAL / "kenya_test_amounts.csv", index=False)
e = np.log(ke.Total_Amount) - final.predict_log(ke)
print(f"limit quantile {LIMIT_QUANTILE:.0%}: offset {final.offset:+.3f} (log), smearing factor {final.smear:.3f}")
print(f"Kenyan Zindi test loans ({len(ke)}): MAE log {np.abs(e).mean():.3f}, within 2x {(np.abs(e) < np.log(2)).mean():.1%}, "
      f"fits under limit {(ke.Total_Amount <= out.limit.values).mean():.1%}")

limit quantile 75%: offset +0.391 (log), smearing factor 1.235
Kenyan Zindi test loans (15069): MAE log 0.573, within 2x 70.8%, fits under limit 70.5%
